***The following notebook runs through how I identified the LMC analogs for the MWest suite based on the properties of the LMC analog in the MWest paper. Note, that SYMLIB has only run the simulations until a scale factor of 1, but the properties in the paper have values for > 1 scale factor. This has been circumvented by looking at the ROCKSTAR files and working backwards to look at the properties, but these LMC analogs are the same as recognized with this python script so has not been inclued***

In [1]:
# Imports
import os
import numpy as np
import symlib
import pandas as pd

# Replace base_dir with your directory containting the SYMLIB files for MWest
base_dir = "/data/darkmatter/programs/symlib/standard/"
suite_name = "MWest"

# This chunk of code verifies the files are being accessed correctly
i_host = 0

sim_dir = symlib.get_host_directory(base_dir, suite_name, i_host)
print("sim_dir =", sim_dir)
print("host_name =", os.path.basename(sim_dir.rstrip("/")))

print("\nsymlib attributes with 'scale' in the name:")
print([x for x in dir(symlib) if "scale" in x.lower()])

h, hist = symlib.read_subhalos(sim_dir)
print("\nh dtype names:")
print(h.dtype.names)

print("\nh shape:", h.shape)

sim_dir = /data/darkmatter/programs/symlib/standard/MWest/Halo004
host_name = Halo004

symlib attributes with 'scale' in the name:
['scale_factors']

h dtype names:
('id', 'mvir', 'vmax', 'rvmax', 'x', 'v', 'ok', 'rvir', 'cvir')

h shape: (402, 236)


***Some helper functions***

In [3]:
def get_scale_array(sim_dir):
    """
    Return the scale-factor array for a given simulation directory.

    Parameters
    ----------
    sim_dir : str
        Path to the simulation directory.

    Returns
    -------
    numpy.ndarray
        Array of cosmological scale factors for the simulation snapshots.
    """
    return np.asarray(symlib.scale_factors(sim_dir))


def nearest_snapshot_to_scale(sim_dir, a_target):
    """
    Find the snapshot index whose scale factor is closest to a target scale factor.

    Parameters
    ----------
    sim_dir : str
        Path to the simulation directory.
    a_target : float
        Target cosmological scale factor.

    Returns
    -------
    int
        Index of the snapshot with scale factor closest to `a_target`.
    """
    sf = get_scale_array(sim_dir)
    return int(np.argmin(np.abs(sf - a_target)))


def get_host_name_from_index(base_dir, suite_name, i_host):
    """
    Return the host halo name corresponding to a given host index.

    Parameters
    ----------
    base_dir : str
        Base directory containing the simulation suite.
    suite_name : str
        Name of the Symphony simulation suite.
    i_host : int
        Index of the host halo within the suite.

    Returns
    -------
    str
        Host halo directory name, for example ``"Halo032"``.
    """
    sim_dir = symlib.get_host_directory(base_dir, suite_name, i_host)
    return os.path.basename(sim_dir.rstrip("/"))

***The following table is a list of properties of the LMC analog of the MWest suite, pulled from this paper- https://arxiv.org/abs/2404.08043***

In [4]:
lmc_targets_by_name = {
    "Halo004": {"a_LMC_50": 1.00, "M_LMC": 1.50e11, "Mpeak_LMC": 1.74e11, "r_LMC": 61.0},
    "Halo113": {"a_LMC_50": 1.01, "M_LMC": 2.80e10, "Mpeak_LMC": 3.60e10, "r_LMC": 49.0},
    "Halo169": {"a_LMC_50": 0.97, "M_LMC": 1.85e11, "Mpeak_LMC": 2.29e11, "r_LMC": 58.0},
    "Halo170": {"a_LMC_50": 0.97, "M_LMC": 2.16e11, "Mpeak_LMC": 2.61e11, "r_LMC": 50.0},
    "Halo222": {"a_LMC_50": 1.05, "M_LMC": 2.25e11, "Mpeak_LMC": 2.67e11, "r_LMC": 63.0},
    "Halo229": {"a_LMC_50": 1.01, "M_LMC": 1.00e10, "Mpeak_LMC": 1.90e10, "r_LMC": 64.0},
    "Halo282": {"a_LMC_50": 1.04, "M_LMC": 5.30e10, "Mpeak_LMC": 7.10e10, "r_LMC": 38.0},
    "Halo327": {"a_LMC_50": 0.94, "M_LMC": 1.16e11, "Mpeak_LMC": 1.41e11, "r_LMC": 44.0},
    "Halo349": {"a_LMC_50": 0.90, "M_LMC": 2.33e11, "Mpeak_LMC": 2.90e11, "r_LMC": 52.0},
    "Halo407": {"a_LMC_50": 1.00, "M_LMC": 7.80e10, "Mpeak_LMC": 9.60e10, "r_LMC": 54.0},
    "Halo453": {"a_LMC_50": 0.96, "M_LMC": 2.99e11, "Mpeak_LMC": 3.30e11, "r_LMC": 57.0},
    "Halo476": {"a_LMC_50": 0.96, "M_LMC": 2.04e11, "Mpeak_LMC": 2.50e11, "r_LMC": 50.0},
    "Halo659": {"a_LMC_50": 0.96, "M_LMC": 3.60e10, "Mpeak_LMC": 8.00e10, "r_LMC": 44.0},
    "Halo666": {"a_LMC_50": 1.00, "M_LMC": 6.12e11, "Mpeak_LMC": 6.36e11, "r_LMC": 79.0},
    "Halo719": {"a_LMC_50": 1.01, "M_LMC": 3.98e11, "Mpeak_LMC": 4.63e11, "r_LMC": 51.0},
    "Halo747": {"a_LMC_50": 0.97, "M_LMC": 5.90e10, "Mpeak_LMC": 7.30e10, "r_LMC": 54.0},
    "Halo756": {"a_LMC_50": 1.00, "M_LMC": 1.34e11, "Mpeak_LMC": 1.61e11, "r_LMC": 66.0},
    "Halo788": {"a_LMC_50": 1.00, "M_LMC": 4.90e10, "Mpeak_LMC": 6.80e10, "r_LMC": 34.0},
    "Halo975": {"a_LMC_50": 0.99, "M_LMC": 2.91e11, "Mpeak_LMC": 3.15e11, "r_LMC": 34.0},
    "Halo983": {"a_LMC_50": 1.00, "M_LMC": 1.80e11, "Mpeak_LMC": 2.50e11, "r_LMC": 49.0},
}

***Main LMC analog finder function***

In [5]:
def identify_lmc_analog_for_host_at_target_snapshot(
    base_dir,
    suite_name,
    i_host,
    target,
    w_m=4.0,
    w_mpeak=2.0,
    w_r=0.1,
    top_n=5,
    verbose=True,
):
    """
    Identify the best LMC analog subhalo for a given host at a target snapshot.

    This function searches through all valid subhalos in a Symphony host halo and
    selects the subhalo whose properties most closely match a target LMC analog.
    The comparison is performed at the snapshot nearest to the target scale factor
    `a_LMC_50`. Candidate subhalos are scored using a weighted distance in
    log-space between their instantaneous mass, peak mass, and galactocentric
    radius and the corresponding target LMC values.

    Parameters
    ----------
    base_dir : str
        Base directory containing the simulation suite.
    suite_name : str
        Name of the Symphony simulation suite.
    i_host : int
        Index of the host halo within the simulation suite.
    target : dict
        Dictionary containing the target LMC analog properties. Required keys are
        ``"a_LMC_50"``, ``"M_LMC"``, ``"Mpeak_LMC"``, and ``"r_LMC"``.
    w_m : float, optional
        Weight applied to the instantaneous virial mass mismatch term.
    w_mpeak : float, optional
        Weight applied to the peak-mass mismatch term.
    w_r : float, optional
        Weight applied to the radial-distance mismatch term.
    top_n : int, optional
        Number of best-scoring candidates to store and optionally print.
    verbose : bool, optional
        If True, print the best match and top candidate subhalos.

    Returns
    -------
    dict or None
        Dictionary containing the selected LMC analog and its matching
        diagnostics. Returns ``None`` if no valid candidates are found.

    Notes
    -----
    The central host halo is excluded from the candidate list. Candidate scores
    are computed in log-space using instantaneous mass, peak mass, and radius.
    Lower scores indicate better LMC analog matches.
    """

    sim_dir = symlib.get_host_directory(base_dir, suite_name, i_host)
    host_name = os.path.basename(sim_dir.rstrip("/"))

    h, hist = symlib.read_subhalos(sim_dir)

    # match at snapshot nearest the tabulated a_LMC_50
    i_match = nearest_snapshot_to_scale(sim_dir, target["a_LMC_50"])
    sf = get_scale_array(sim_dir)
    a_match = sf[i_match]

    ok_match = h["ok"][:, i_match].copy()
    ok_match[0] = False  # exclude host

    x_match = h[:, i_match]["x"]
    r_match = np.sqrt(np.sum(x_match**2, axis=1))
    m_match = h[:, i_match]["mvir"]
    mpeak = hist["mpeak"]

    cand = np.where(ok_match)[0]
    if len(cand) == 0:
        if verbose:
            print(f"{host_name}: no valid candidates at snapshot {i_match}")
        return None

    eps = 1e-300
    logm   = np.log10(np.maximum(m_match[cand], eps))
    logmp  = np.log10(np.maximum(mpeak[cand], eps))
    logr   = np.log10(np.maximum(r_match[cand], eps))

    logm_t  = np.log10(target["M_LMC"])
    logmp_t = np.log10(target["Mpeak_LMC"])
    logr_t  = np.log10(target["r_LMC"])

    score = (
        w_m     * (logm  - logm_t )**2 +
        w_mpeak * (logmp - logmp_t)**2 +
        w_r     * (logr  - logr_t )**2
    )

    order = np.argsort(score)
    best_idx = cand[order[0]]

    top_rows = []
    for j in order[:top_n]:
        idx = cand[j]
        top_rows.append({
            "idx": int(idx),
            "M_at_match": float(m_match[idx]),
            "Mpeak": float(mpeak[idx]),
            "r_at_match": float(r_match[idx]),
            "score": float(score[j]),
        })

    result = {
        "host_index": int(i_host),
        "host_name": host_name,
        "lmc_idx": int(best_idx),
        "i_match": int(i_match),
        "a_target": float(target["a_LMC_50"]),
        "a_match": float(a_match),
        "target_M_at_match": float(target["M_LMC"]),
        "target_Mpeak": float(target["Mpeak_LMC"]),
        "target_r_at_match": float(target["r_LMC"]),
        "matched_M_at_match": float(m_match[best_idx]),
        "matched_Mpeak": float(mpeak[best_idx]),
        "matched_r_at_match": float(r_match[best_idx]),
        "best_score": float(score[order[0]]),
        "top_candidates": top_rows,
    }

    if verbose:
        print(f"{host_name}: best LMC idx = {best_idx}")
        print(f"  target a={target['a_LMC_50']:.3f}, matched snapshot={i_match}, matched a={a_match:.3f}")
        print(f"  target   M_at_match={target['M_LMC']:.3e}, Mpeak={target['Mpeak_LMC']:.3e}, r={target['r_LMC']:.1f}")
        print(f"  matched  M_at_match={m_match[best_idx]:.3e}, Mpeak={mpeak[best_idx]:.3e}, r={r_match[best_idx]:.1f}")
        print(f"  score={score[order[0]]:.4e}")
        print("  top candidates:")
        for row in top_rows:
            print(
                f"    idx={row['idx']:3d}  "
                f"M_at_match={row['M_at_match']:.3e}  "
                f"Mpeak={row['Mpeak']:.3e}  "
                f"r_at_match={row['r_at_match']:.1f}  "
                f"score={row['score']:.4e}"
            )

    return result

***Running the LMC analog function***

In [7]:
lmc_index_by_host = {}
match_summaries = []

for i_host in range(symlib.n_hosts(suite_name)):
    host_name = get_host_name_from_index(base_dir, suite_name, i_host)

    if host_name not in lmc_targets_by_name:
        print(f"Skipping host {i_host}: {host_name} not in lmc_targets_by_name")
        continue

    result = identify_lmc_analog_for_host_at_target_snapshot(
        base_dir=base_dir,
        suite_name=suite_name,
        i_host=i_host,
        target=lmc_targets_by_name[host_name],
        w_m=4.0,
        w_mpeak=2.0,
        w_r=0.1,
        top_n=5,
        verbose=True,
    )

    if result is not None:
        lmc_index_by_host[i_host] = result["lmc_idx"]
        match_summaries.append({
            "host_index": result["host_index"],
            "host_name": result["host_name"],
            "lmc_idx": result["lmc_idx"],
            "i_match": result["i_match"],
            "a_target": result["a_target"],
            "a_match": result["a_match"],
            "target_M_at_match": result["target_M_at_match"],
            "matched_M_at_match": result["matched_M_at_match"],
            "target_Mpeak": result["target_Mpeak"],
            "matched_Mpeak": result["matched_Mpeak"],
            "target_r_at_match": result["target_r_at_match"],
            "matched_r_at_match": result["matched_r_at_match"],
            "best_score": result["best_score"],
        })



Halo004: best LMC idx = 1
  target a=1.000, matched snapshot=235, matched a=1.000
  target   M_at_match=1.500e+11, Mpeak=1.740e+11, r=61.0
  matched  M_at_match=1.509e+11, Mpeak=1.704e+11, r=60.9
  score=1.8680e-04
  top candidates:
    idx=  1  M_at_match=1.509e+11  Mpeak=1.704e+11  r_at_match=60.9  score=1.8680e-04
    idx=  3  M_at_match=1.621e+10  Mpeak=3.110e+10  r_at_match=79.1  score=4.8538e+00
    idx=  4  M_at_match=1.258e+10  Mpeak=1.980e+10  r_at_match=186.2  score=6.4395e+00
    idx=  7  M_at_match=7.636e+09  Mpeak=1.203e+10  r_at_match=225.8  score=9.4145e+00
    idx=  9  M_at_match=5.719e+09  Mpeak=1.058e+10  r_at_match=42.5  score=1.1012e+01
Halo113: best LMC idx = 5
  target a=1.010, matched snapshot=235, matched a=1.000
  target   M_at_match=2.800e+10, Mpeak=3.600e+10, r=49.0
  matched  M_at_match=3.040e+10, Mpeak=3.633e+10, r=77.3
  score=9.0443e-03
  top candidates:
    idx=  5  M_at_match=3.040e+10  Mpeak=3.633e+10  r_at_match=77.3  score=9.0443e-03
    idx= 12  M_a

In [8]:
match_df = pd.DataFrame(match_summaries)
display(match_df)

print("\nFinal lmc_index_by_host:")
print(lmc_index_by_host)

,host_index,host_name,lmc_idx,i_match,a_target,a_match,target_M_at_match,matched_M_at_match,target_Mpeak,matched_Mpeak,target_r_at_match,matched_r_at_match,best_score
0,0,Halo004,1,235,1.00,1.000000,1.500000e+11,1.508572e+11,1.740000e+11,1.704286e+11,61.0,60.892204,1.867958e-04
1,1,Halo113,5,235,1.01,1.000000,2.800000e+10,3.040000e+10,3.600000e+10,3.632857e+10,49.0,77.257866,9.044297e-03
2,2,Halo169,1,233,0.97,0.974827,1.850000e+11,1.854286e+11,2.290000e+11,2.288572e+11,58.0,58.150070,4.315067e-06
3,3,Halo170,1,233,0.97,0.974827,2.160000e+11,2.170000e+11,2.610000e+11,2.667143e+11,50.0,49.852409,1.932073e-04
4,4,Halo222,1,235,1.05,1.000000,2.250000e+11,2.642857e+11,2.670000e+11,2.654286e+11,63.0,136.650330,3.086036e-02
5,5,Halo229,7,235,1.01,1.000000,1.000000e+10,1.051286e+10,1.900000e+10,1.865714e+10,64.0,91.379829,4.404695e-03
6,6,Halo282,4,235,1.04,1.000000,5.300000e+10,6.205714e+10,7.100000e+10,7.081429e+10,38.0,149.547623,5.418263e-02
7,7,Halo327,2,230,0.94,0.938251,1.160000e+11,1.145000e+11,1.410000e+11,1.412143e+11,44.0,44.095154,1.287617e-04
8,8,Halo349,1,227,0.90,0.903046,2.330000e+11,2.324286e+11,2.900000e+11,2.875714e+11,52.0,52.241650,3.162420e-05
9,9,Halo407,2,235,1.00,1.000000,7.800000e+10,7.817143e+10,9.600000e+10,9.585715e+10,54.0,53.567383,5.694644e-06



Final lmc_index_by_host:
{0: 1, 1: 5, 2: 1, 3: 1, 4: 1, 5: 7, 6: 4, 7: 2, 8: 1, 9: 2, 10: 1, 11: 1, 12: 2, 13: 1, 14: 1, 15: 6, 16: 3, 17: 3, 18: 1, 19: 1}
